<a href="https://colab.research.google.com/github/camille2019/Women-Capital-Trial-Analysis/blob/main/propensity_matching_psm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install psmpy


In [2]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

from psmpy import PsmPy
from psmpy.functions import cohenD
from psmpy.plotting import *

In [379]:
from psmpy import PsmPy


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
save_path = '/content/drive/MyDrive/Capital_trials/'
# full_meta = pd.read_csv(save_path + 'full_crime_meta_.csv')
# ohe_meta = pd.read_csv('/content/drive/MyDrive/Capital_trials/ohe_metadata_only.csv')

ohe_meta = pd.read_csv('/content/updated_full_metadata_ohe.csv')

In [28]:
ohe_meta

ohe_meta = ohe_meta.drop('Unnamed: 0', axis=1)


In [29]:
ohe_meta.columns

Index(['Name', 'crime_cont_threat', 'crime_disregard_human_life',
       'crime_manner_mayhem', 'crime_retaliation', 'crime_old_disabled_victim',
       'crime_weapon_arson', 'crime_weapon', 'crime_aggrevated_assult',
       'crime_attempted_rape', 'crime_child_victim', 'crime_public_risk',
       'crime_premeditated', 'crime_multiple_felonies',
       'crime_especially_cruel', 'crime_gang_murder', 'crime_bodily_injury',
       'crime_interfere_arrest', 'crime_kidnapping', 'crime_larceny',
       'crime_lewd_acts_child', 'crime_lying_in_wait', 'crime_police_victim',
       'crime_handgun', 'crime_weapon_poison', 'crime_prior_conv',
       'crime_prior_off', 'crime_rape', 'crime_sexual_assult',
       'crime_shooting_car', 'crime_sodomy', 'crime_torture',
       'crime_unlawful_firearm', 'victim_list', 'victim_child',
       'victim_child_care', 'victim_partner', 'victim_ex_partner',
       'victim_stranger', 'victim_family', 'victim_familiar', 'State_AL',
       'State_AZ', 'State_CA',

In [191]:
ohe_meta.loc[ohe_meta['crime_manner_mayhem'] == True, ['Name', 'gender_woman']]


,Name,gender_woman
9,"Matthews, Damon Roshun",False
18,"Johnson, Dexter Darnell",False
113,Belinda Magaña,True
154,Veronica Gonzales,True


In [387]:
def add_propensity(propensity_columns, treatment_column, df_encoded, df_meta, matched_group_name, add_score_col= True, cal = None, rep=False,  h_m = 1, m= None):

  psm_df = df_encoded[propensity_columns + [treatment_column]].copy()


  psm_df = psm_df.reset_index(drop=True)
  psm_df["row_id"] = psm_df.index

  psm = PsmPy(psm_df, treatment=treatment_column, indx="row_id", exclude=[])

  psm.logistic_ps(balance=True)

  if h_m > 1:
    if m is None:
      psm.kdtree_matched_12n(matcher='propensity_logit', how_many = h_m )
    else:
      psm.kdtree_matched_12n(matcher= m, how_many = h_m )


  else:
    if m is None:
      psm.kdtree_matched(matcher='propensity_logit', caliper=cal, replacement = rep )
    else:
      psm.kdtree_matched(matcher=m, caliper=cal, replacement = rep )


  pred = psm.predicted_data

  df_meta = df_meta.reset_index(drop=True)

  scores_col_name = matched_group_name + '_prop_scores'

  if add_score_col:

    score_map = pred[["row_id", "propensity_score"]].drop_duplicates("row_id")
    df_meta = df_meta.merge(score_map, left_index=True, right_on="row_id", how="left")
    df_meta = df_meta.drop(columns=["row_id"])
    df_meta = df_meta.rename(columns={"propensity_score": matched_group_name + "_prop_scores"})

  # psm.kdtree_matched(matcher="propensity_score", replacement=False, caliper=None)
  # matched_df = psm.df_matched
  # matched_ids = set(psm.df_matched["row_id"].unique())
  # df_meta = df_meta.reset_index(drop=True)

  # binary_col_name = 'psm_' + matched_group_name

  # df_meta[binary_col_name] = df_meta.index.isin(matched_ids)
  matched_ids = set(psm.df_matched["row_id"].unique())
  df_meta["psm_" + matched_group_name] = df_meta.index.isin(matched_ids)

  return df_meta



In [44]:
race_cols =['Race_Asian',
       'Race_Black', 'Race_Latinx', 'Race_Native American', 'Race_White',]

In [45]:
state_cols = ['State_AL',
       'State_AZ', 'State_CA', 'State_FL', 'State_GA', 'State_ID', 'State_KY',
       'State_LA', 'State_MS', 'State_NC', 'State_OH', 'State_OK', 'State_PA',
       'State_SC', 'State_TN', 'State_TX']

In [46]:
crime_cols = ['crime_cont_threat', 'crime_disregard_human_life',
       'crime_manner_mayhem', 'crime_retaliation', 'crime_old_disabled_victim',
       'crime_weapon_arson', 'crime_weapon', 'crime_aggrevated_assult',
       'crime_attempted_rape', 'crime_child_victim', 'crime_public_risk',
       'crime_premeditated', 'crime_multiple_felonies',
       'crime_especially_cruel', 'crime_gang_murder', 'crime_bodily_injury',
       'crime_interfere_arrest', 'crime_kidnapping', 'crime_larceny',
       'crime_lewd_acts_child', 'crime_lying_in_wait', 'crime_police_victim',
       'crime_handgun', 'crime_weapon_poison', 'crime_prior_conv',
       'crime_prior_off', 'crime_rape', 'crime_sexual_assult',
       'crime_shooting_car', 'crime_sodomy', 'crime_torture',
       'crime_unlawful_firearm', ]

In [47]:
victim_relationship_cols = ['victim_child',
       'victim_child_care', 'victim_partner', 'victim_ex_partner',
       'victim_stranger', 'victim_family', 'victim_familiar']

In [48]:
sentence_band_cols  = [ 'sentence_band_1980-1984', 'sentence_band_1985-1989',
       'sentence_band_1990-1994', 'sentence_band_1995-1999',
       'sentence_band_2000-2004', 'sentence_band_2005-2009',
       'sentence_band_2010-2014', 'sentence_band_2015-2019',
       'sentence_band_2020-2024', 'sentence_band_2070-2074']

In [49]:
state_race_crime_cols = state_cols + race_cols + crime_cols

In [50]:
race_crime_cols = race_cols + crime_cols

In [193]:
df = pd.DataFrame()
df['def_name'] = ohe_meta['Name']
df['def_woman'] = ohe_meta['gender_woman']

In [226]:
def find_caliper(updated_df, propensity_score_col):
  updated_df['logit_ps'] = np.log(updated_df[propensity_score_col] / (1 - updated_df[propensity_score_col]))

  std_logit = updated_df['logit_ps'].std()

  caliper = 0.2 * std_logit

  tight_caliper = 0.1 * std_logit

  loose_caliper = 0.5 * std_logit

  print(f"Recommended Caliper (.2 * std): {caliper}")
  print(f"Tight Caliper (.1 * std): {tight_caliper}")
  print(f"Loose Caliper (.5 * std): {loose_caliper}")

In [104]:
def check_calipers_same_match(updated_df, match_version1, match_version2):
  version1 = set(updated_df.loc[updated_df[match_version1] == True, 'def_name'].to_list() )
  version2 = set(updated_df.loc[updated_df[match_version2] == True, 'def_name'].to_list() )
  groups_equal = state_race_crime_calp25 == state_race_crime_default_calp
  return groups_equal

# Propensity matching for Crime alone


In [386]:
ohe_meta.columns

Index(['Name', 'crime_cont_threat', 'crime_disregard_human_life',
       'crime_manner_mayhem', 'crime_retaliation', 'crime_old_disabled_victim',
       'crime_weapon_arson', 'crime_weapon', 'crime_aggrevated_assult',
       'crime_attempted_rape', 'crime_child_victim', 'crime_public_risk',
       'crime_premeditated', 'crime_multiple_felonies',
       'crime_especially_cruel', 'crime_gang_murder', 'crime_bodily_injury',
       'crime_interfere_arrest', 'crime_kidnapping', 'crime_larceny',
       'crime_lewd_acts_child', 'crime_lying_in_wait', 'crime_police_victim',
       'crime_handgun', 'crime_weapon_poison', 'crime_prior_conv',
       'crime_prior_off', 'crime_rape', 'crime_sexual_assult',
       'crime_shooting_car', 'crime_sodomy', 'crime_torture',
       'crime_unlawful_firearm', 'victim_list', 'victim_child',
       'victim_child_care', 'victim_partner', 'victim_ex_partner',
       'victim_stranger', 'victim_family', 'victim_familiar', 'State_AL',
       'State_AZ', 'State_CA',

In [390]:
crime_only_prop = add_propensity(crime_cols, 'gender_woman', ohe_meta, df, 'crime_only_match')


In [391]:
crime_only_prop

,def_name,def_woman,crime_only_match_prop_scores,psm_crime_only_match
47,"Gaxiola, Albert Robert",False,0.539457,True
48,"Reid, Albert Ezron",False,0.517266,False
49,"Lackey, Andrew",False,0.494814,False
50,"Milam, Blaine Keith",False,0.506324,False
51,"Kennedy, Carlos",False,0.483225,False
...,...,...,...,...
42,Tina Brown,True,0.781554,False
43,Valerie Martin,True,0.639901,True
44,Veronica Gonzales,True,0.595728,False
45,Virginia Caudill,True,0.499680,True


In [392]:
find_caliper(crime_only_prop, 'crime_only_match_prop_scores')

Recommended Caliper (.2 * std): 0.10177805244148999
Tight Caliper (.1 * std): 0.050889026220744996
Loose Caliper (.5 * std): 0.25444513110372496


In [393]:
crime_only_prop = add_propensity(crime_cols, 'gender_woman',  ohe_meta, crime_only_prop, 'crime_only_match_cal_rec', cal=0.10177805244148999, add_score_col=False , m ='propensity_score')
crime_only_prop = add_propensity(crime_cols, 'gender_woman', ohe_meta, crime_only_prop, 'crime_only_match_cal_tight', cal=0.050889026220744996, add_score_col=False,  m ='propensity_score')
crime_only_prop = add_propensity(crime_cols, 'gender_woman', ohe_meta, crime_only_prop, 'crime_only_match_cal_loose', cal=0.25444513110372496, add_score_col=False,  m ='propensity_score')

crime_only_prop = add_propensity(crime_cols, 'gender_woman', ohe_meta, crime_only_prop, 'crime_only_match_cal_tight_replacement', cal=0.050889026220744996, add_score_col=False, rep = True,  m ='propensity_score')
crime_only_prop = add_propensity(crime_cols, 'gender_woman', ohe_meta, crime_only_prop, 'crime_only_match_cal_loose_replacement', cal=0.25444513110372496, add_score_col=False, rep= True,  m ='propensity_score')


/usr/local/lib/python3.12/dist-packages/psmpy/psmpy.py:374: UserWarning: Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False
  warnings.warn('Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False')
/usr/local/lib/python3.12/dist-packages/psmpy/psmpy.py:374: UserWarning: Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False
  warnings.warn('Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (eff

In [394]:
crime_only_prop

,def_name,def_woman,crime_only_match_prop_scores,psm_crime_only_match,logit_ps,psm_crime_only_match_cal_rec,psm_crime_only_match_cal_tight,psm_crime_only_match_cal_loose,psm_crime_only_match_cal_tight_replacement,psm_crime_only_match_cal_loose_replacement
0,"Gaxiola, Albert Robert",False,0.539457,True,0.158157,True,True,True,False,False
1,"Reid, Albert Ezron",False,0.517266,False,0.069090,True,False,True,False,False
2,"Lackey, Andrew",False,0.494814,False,-0.020746,False,False,True,False,False
3,"Milam, Blaine Keith",False,0.506324,False,0.025296,False,False,True,False,False
4,"Kennedy, Carlos",False,0.483225,False,-0.067127,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
152,Tina Brown,True,0.781554,False,1.274746,False,False,False,True,True
153,Valerie Martin,True,0.639901,True,0.574932,False,False,True,True,True
154,Veronica Gonzales,True,0.595728,False,0.387697,True,False,True,True,True
155,Virginia Caudill,True,0.499680,True,-0.001280,True,True,True,True,True


In [398]:
are_equal = crime_only_prop['psm_crime_only_match_cal_rec'].equals(crime_only_prop['psm_crime_only_match_cal_tight_replacement'])
are_equal

False

In [399]:
crime_only_prop.to_csv('crime_matched_group_options.csv')

#Propensity matching for state + race+ crime